In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
import os

pd.set_option("display.max_columns", None)

print("Prétraitement - Bibliothèques chargées")

In [ ]:
print("Chargement et fusion des données brutes...")
df_sensor = pd.read_csv('Projet_Data/application.csv')
df_network = pd.read_csv('Projet_Data/lora.csv')

df_sensor['time'] = pd.to_datetime(df_sensor['time'], unit='ms')
df_network['time'] = pd.to_datetime(df_network['time'], unit='ms')

df = pd.merge(df_sensor, df_network, on=['time', 'deveui'], how='inner')
df = df.sort_values("time").reset_index(drop=True)

print(f"Dimensions initiales : {df.shape}")
display(df.head())

In [ ]:
print("Création des variables temporelles...")
df["hour"] = df["time"].dt.hour
df["minute"] = df["time"].dt.minute
df["day"] = df["time"].dt.day
df["month"] = df["time"].dt.month
df["day_of_week"] = df["time"].dt.dayofweek
df["week"] = df["time"].dt.isocalendar().week.astype(int)

display(df[['time', 'hour', 'minute', 'day', 'month', 'day_of_week', 'week']].head())

In [ ]:
target_cols = ['temperature', 'humidity', 'co2']
targets = ['target_temp', 'target_humidity', 'target_co2']

for col, target_name in zip(target_cols, targets):
    df[target_name] = df[col].shift(-1)

print("Création des variables cibles décalées :")
display(df[["time"] + target_cols + targets].tail())

In [ ]:
print("Nettoyage des valeurs aberrantes (Outliers)...")
# Remplacer les températures impossibles par NaN
if 'temperature' in df.columns:
    df.loc[(df['temperature'] > 60) | (df['temperature'] < -20), 'temperature'] = np.nan

# Remplacer les niveaux de CO2 impossibles par NaN
if 'co2' in df.columns:
    df.loc[df['co2'] > 10000, 'co2'] = np.nan

# L'humidité est déjà capée naturellement entre 0 et 100, mais on peut s'en assurer :
if 'humidity' in df.columns:
    df.loc[(df['humidity'] > 100) | (df['humidity'] < 0), 'humidity'] = np.nan

In [ ]:
print("Traitement des valeurs manquantes...")
feature_cols_to_impute = [col for col in df.columns if col not in ['time', 'deveui'] + targets]
numeric_cols = df[feature_cols_to_impute].select_dtypes(include=["float64", "int64", "int32"]).columns

imputer = SimpleImputer(strategy='median')
df[numeric_cols] = imputer.fit_transform(df[numeric_cols])

print("Valeurs manquantes après imputation :")
display(df[numeric_cols].isnull().sum().sort_values(ascending=False).head(5))

In [ ]:
df = df.dropna(subset=targets)
print(f"Dimensions après suppression des cibles NaN : {df.shape}")

In [ ]:
excluded_columns = ["time", "deveui"] + targets
feature_columns = [col for col in df.columns if col not in excluded_columns]

print(f"Nombre de features : {len(feature_columns)}")
print(f"Nombre de cibles : {len(targets)}")

In [ ]:
os.makedirs("Projet_Data/processed", exist_ok=True)
output_path = "Projet_Data/processed/iot_preprocessed.csv"

df.to_csv(output_path, index=False)
print(f"Dataset multi-sorties sauvegardé avec succès sous : {output_path}")